# Milestone 1 lab, notebook 7 of 8: Six questions for any table

## Where we are

| Notebook | What you do |
|---|---|
| 1. The question and five green checks | Meet Lotus Retails and the Monday question. Run the given pipeline for one night. Read its five checks |
| 2. Airflow runs it | Let Airflow run the same night. Read the DAG: its tasks and its arrows |
| 3. The number by hand | Add up one store's receipts yourself. Find out why the two numbers differ |
| 4. Who is hurt | For each table: who uses it, and what happens to them when it is wrong |
| 5. The check card | Meet the page you fill in before any check. Say what one row means |
| 6. Read the table first | Count the rows, find the smallest and largest values, count the empty cells |
| **7. Six questions for any table** (you are here) | Sort your worries under six questions. See which ones the five checks answer |
| 8. The production version | The same logic as a tested program |

**About ten minutes.**

In [1]:
import pandas as pd

from lab_helpers import save_result, sql

pd.set_option("display.max_colwidth", 80)

## Step 1: Six questions

You have a check card with ranked worries, and four candidates from profiling. Is the list complete? How would you know?

There are six questions you can ask of any table. They are a way to see what you forgot.

| Question | It asks | At work it is called |
|---|---|---|
| **Complete?** | Is anything missing? | Completeness |
| **Unique?** | Is one thing stored twice? | Uniqueness |
| **Valid?** | Is each value allowed? | Validity |
| **Accurate?** | Does it match the real world? | Accuracy |
| **Consistent?** | Do two tables agree? | Consistency |
| **Fresh?** | Is it recent enough? | Timeliness |

Together they are called the **six dimensions** of data quality.

## Step 2: Put the five standard checks under them

Start with what Lotus already has.

In [2]:
standard = pd.DataFrame(
    [
        (1, "The table is not empty", "Complete?"),
        (2, "The key is not null", "Complete?"),
        (3, "The key is unique", "Unique?"),
        (4, "The status is an allowed value", "Valid?"),
        (5, "The total is positive", "Valid?"),
    ],
    columns=["#", "standard check", "question"],
)
standard

,#,standard check,question
0,1,The table is not empty,Complete?
1,2,The key is not null,Complete?
2,3,The key is unique,Unique?
3,4,The status is an allowed value,Valid?
4,5,The total is positive,Valid?


## Step 3: Put your candidates under them

These come from the check card (notebook 5) and from profiling (notebook 6). Each one names what must be prepared first.

In [3]:
candidates = [
    {"question": "Complete?", "worry": "A store's whole day is missing",
     "candidate": "Every open store sent a file for the day", "prepare": "Count the files per store per day"},
    {"question": "Complete?", "worry": "A day is far too small or far too big",
     "candidate": "The lines of a store's day are near a normal day", "prepare": "A normal day per store and weekday"},
    {"question": "Unique?", "worry": "The same sale counted twice",
     "candidate": "receipt_id + line_no is there once", "prepare": "Group by the true key, not by line_id"},
    {"question": "Valid?", "worry": "Money with the wrong sign",
     "candidate": "A refund's amount is below zero", "prepare": "Read sales and refunds apart"},
    {"question": "Valid?", "worry": "Money of the wrong size",
     "candidate": "A quantity is believable for its product", "prepare": "A range per product"},
    {"question": "Valid?", "worry": "A refund that refunds nothing",
     "candidate": "A refund names its receipt", "prepare": "Nothing"},
    {"question": "Accurate?", "worry": "The lines do not match the money in the till",
     "candidate": "The lines of a till add up to the till's own total", "prepare": "Add up per till per day, join to till_totals"},
    {"question": "Consistent?", "worry": "A line the join drops",
     "candidate": "Every sale is for a product that exists", "prepare": "Compare raw rows with clean rows"},
    {"question": "Consistent?", "worry": "A sale on the wrong day",
     "candidate": "A sale is inside the store's opening hours", "prepare": "Join to stores"},
    {"question": "Fresh?", "worry": "A file that arrives late",
     "candidate": "Every file arrived before 01:00", "prepare": "The arrival time of each file"},
]
pd.DataFrame(candidates)

,question,worry,candidate,prepare
0,Complete?,A store's whole day is missing,Every open store sent a file for the day,Count the files per store per day
1,Complete?,A day is far too small or far too big,The lines of a store's day are near a normal day,A normal day per store and weekday
2,Unique?,The same sale counted twice,receipt_id + line_no is there once,"Group by the true key, not by line_id"
3,Valid?,Money with the wrong sign,A refund's amount is below zero,Read sales and refunds apart
4,Valid?,Money of the wrong size,A quantity is believable for its product,A range per product
5,Valid?,A refund that refunds nothing,A refund names its receipt,Nothing
6,Accurate?,The lines do not match the money in the till,The lines of a till add up to the till's own total,"Add up per till per day, join to till_totals"
7,Consistent?,A line the join drops,Every sale is for a product that exists,Compare raw rows with clean rows
8,Consistent?,A sale on the wrong day,A sale is inside the store's opening hours,Join to stores
9,Fresh?,A file that arrives late,Every file arrived before 01:00,The arrival time of each file


## Step 4: Count both lists, question by question

In [4]:
questions = ["Complete?", "Unique?", "Valid?", "Accurate?", "Consistent?", "Fresh?"]
tally = pd.DataFrame(
    {
        "standard checks": [int((standard["question"] == q).sum()) for q in questions],
        "your candidates": [sum(c["question"] == q for c in candidates) for q in questions],
    },
    index=questions,
)
tally

,standard checks,your candidates
Complete?,2,2
Unique?,1,1
Valid?,2,3
Accurate?,0,1
Consistent?,0,2
Fresh?,0,1


Read the left column. The five standard checks answer three of the six questions. For **Accurate?**, **Consistent?** and **Fresh?** Lotus has nothing at all.

And the two "Complete?" checks are weak. "Not empty" passes when five stores of six report. "Key not null" says nothing about a missing day.

## Step 5: The question a table cannot answer about itself

Look at **Accurate?**: does it match the real world?

No query on `sales_lines` alone can answer that. The row with +949 is a perfectly formed row. To know it is wrong, you need a second source that saw the same money.

Lotus has one. It is the table nobody reads.

In [5]:
sql("""
    SELECT lines.store_id,
           lines.lines_total,
           tills.tills_total,
           lines.lines_total - tills.tills_total AS difference
    FROM (
        SELECT store_id, sum(amount_cents) AS lines_total
        FROM clean.sales_lines
        WHERE sales_date = '2026-08-24'
        GROUP BY store_id
    ) lines
    JOIN (
        SELECT store_id, sum(net_total_cents) AS tills_total
        FROM clean.till_totals
        WHERE business_date = '2026-08-24'
        GROUP BY store_id
    ) tills ON tills.store_id = lines.store_id
    ORDER BY lines.store_id
""")

,store_id,lines_total,tills_total,difference
0,S01,79524,79524,0
1,S02,90128,90128,0
2,S03,79618,77720,1898
3,S04,57977,57977,0
4,S05,47418,47418,0
5,S06,59519,59519,0


Five stores agree to the cent. One is 1,898 cents apart: $18.98, in `S03`.

That one query would have found the wrong number in **every** store, on the first night, with no luck. Notice what it needed first: two totals and a join. It is not a one-line check.

It is also not a check yet. It has no limit, nobody decided what happens when it fails, and the pipeline does not run it. It is a candidate.

Save the candidates. Milestone 2 turns them into checks.

In [6]:
save_result("candidate_checks", candidates)

Saved notebooks/results/candidate_checks.json


## What you now know

- Six questions for any table: is it complete, unique, valid, accurate, consistent and fresh? They are the **six dimensions** of data quality.
- The five standard checks answer three of them, weakly.
- "Accurate?" needs a second source. A table cannot prove itself.
- You have ten candidate checks, each with a reason, and each sorted under a question.

**Say it in an interview:** "I sort a table's checks under six questions: complete, unique, valid, accurate, consistent, fresh. Our five standard checks covered three. Nothing tested accuracy, and accuracy cannot be tested from the table itself. We compared the sales lines with the tills' own totals, and one store was $18.98 apart."

**The theory:** [Data Quality & Validation](https://www.d8loop.com/modules/knowledge-corner/data-pipeline/data-quality-and-validation), section "What is good data?".

Next: [notebook 8](08_the_production_version.ipynb). The same logic as a tested program.